# LLM evaluation — interview prep

This notebook pairs a **study plan** with **hands-on code** you can extend. Interview loops often expect you to narrate an end-to-end pipeline *and* implement a metric or judge wrapper.

---

## Study plan (2–3 weeks, adjustable)

| Week | Focus | Deliverable |
|------|--------|-------------|
| **1** | **Frame the eval** — task type (QA, summarization, tool-use, RAG), success criteria, failure modes, baseline (human / simpler model). | 1-page spec: inputs, outputs, rubric bullets, and 20–50 golden examples. |
| **1** | **Deterministic metrics** — exact match, token F1, regex constraints, JSON schema validity. | Implement from scratch + explain when they break for generative tasks. |
| **2** | **N-gram / overlap** — BLEU, ROUGE, METEOR; know limitations for LLM text. | Run `evaluate` or `rouge_score`; compare to reference answers. |
| **2** | **Semantic similarity** — SentenceTransformers + cosine; **BERTScore** (token-level); threshold tradeoffs. | Pair predictions with references; discuss false positives on paraphrases. |
| **2** | **LLM-as-judge** — rubric prompts, position bias, scaling, parsing structured scores. | Implement judge prompt + JSON parsing; **always** cross-check a subset with humans. |
| **3** | **Aggregation & rigor** — stratified sampling, confidence intervals, versioning datasets. | Bootstrap CIs; track prompt/dataset git hashes or LangSmith-style runs. |
| **3** | **Ecosystem** — **RAG**: Ragas; **general**: DeepEval, OpenAI Evals patterns, tracing (LangSmith). | One notebook cell each: install + minimal API usage (when you have keys). |

**End-to-end story (say this in interviews):** define task → build/curate golden set → automated checks → semantic / judge layers → human audit on disagreements → aggregate with uncertainty → regression on every change.

---

## Optional installs (run what you need)

```bash
pip install numpy datasets evaluate rouge-score sacrebleu nltk sentence-transformers bert-score python-dotenv openai ragas deepeval
```

Core sections below run with **numpy only**; later cells note optional imports.

## 1. Define the task & toy dataset

Golden examples: `(question, reference_answer, model_prediction)`. Swap `prediction` for your model later.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any


@dataclass
class EvalExample:
    id: str
    question: str
    reference: str
    prediction: str
    metadata: dict[str, Any] | None = None


EXAMPLES: list[EvalExample] = [
    EvalExample(
        id="capitals-1",
        question="What is the capital of France?",
        reference="Paris",
        prediction="Paris",
    ),
    EvalExample(
        id="capitals-2",
        question="Capital of Japan?",
        reference="Tokyo",
        prediction="The capital is Tokyo.",
    ),
    EvalExample(
        id="math-1",
        question="What is 12 + 28?",
        reference="40",
        prediction="41",
    ),
]

len(EXAMPLES), EXAMPLES[0]

(3,
 EvalExample(id='capitals-1', question='What is the capital of France?', reference='Paris', prediction='Paris', metadata=None))

## 2. Normalization + exact match + token F1

Interview classic: implement **EM** and **token-level F1** (SQuAD-style). Discuss casing, punctuation, and articles.

In [24]:
import re as _re
import string


def normalize_answer(s: str) -> str:
    s = s.lower()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = _re.sub(r"\s+", " ", s).strip()
    return s


def exact_match(pred: str, ref: str) -> bool:
    return normalize_answer(pred) == normalize_answer(ref)


def token_f1(pred: str, ref: str) -> float:
    pred_toks = normalize_answer(pred).split()
    ref_toks = normalize_answer(ref).split()
    if not pred_toks or not ref_toks:
        return float(pred_toks == ref_toks)
    common = 0
    ref_counts: dict[str, int] = {}
    for t in ref_toks:
        ref_counts[t] = ref_counts.get(t, 0) + 1
    pred_counts: dict[str, int] = {}
    for t in pred_toks:
        pred_counts[t] = pred_counts.get(t, 0) + 1
    for t, c in pred_counts.items():
        if t in ref_counts:
            common += min(c, ref_counts[t])
    precision = common / len(pred_toks)
    recall = common / len(ref_toks)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


for ex in EXAMPLES:
    print(
        ex.id,
        "EM",
        exact_match(ex.prediction, ex.reference),
        "F1",
        round(token_f1(ex.prediction, ex.reference), 3),
    )

capitals-1 EM True F1 1.0
capitals-2 EM False F1 0.4
math-1 EM False F1 0.0


## 3. ROUGE, BLEU & METEOR via Hugging Face `evaluate`

**ROUGE:** `pip install evaluate rouge-score` (backend for `evaluate.load("rouge")`).

**BLEU:** `evaluate.load("bleu")` uses **`sacrebleu`** under the hood—install may happen automatically on first load; if load fails, run `pip install sacrebleu`.

**METEOR:** `evaluate.load("meteor")` uses **NLTK** (WordNet-based synonym matching). Install `nltk` and ensure corpora download when prompted, or run `nltk.download("wordnet")` and `nltk.download("omw-1.4")`.

METEOR often rewards **paraphrases** more than raw n-gram BLEU because of stemming/synonyms; all three are still weak alone for factual QA.

In [16]:
try:
    import evaluate

    rouge = evaluate.load("rouge")
except Exception as e:
    rouge = None
    print("Skipping ROUGE:", e)

if rouge is not None:
    ex = EXAMPLES[1]
    scores = rouge.compute(
        predictions=[ex.prediction],
        references=[ex.reference],
        use_stemmer=True,
    )
    scores

In [18]:
scores

{'rouge1': np.float64(0.4),
 'rouge2': np.float64(0.0),
 'rougeL': np.float64(0.4),
 'rougeLsum': np.float64(0.4)}

In [19]:
try:
    import evaluate

    bleu = evaluate.load("bleu")
except Exception as e:
    bleu = None
    print("Skipping BLEU:", e)

if bleu is not None:
    # References must be a list of *lists*: each prediction has one or more valid reference strings.
    pred = "The cat sat on the mat"
    ref_para = "The cat is sitting on the mat"

    paraphrase = bleu.compute(predictions=[pred], references=[[ref_para]])
    identical = bleu.compute(predictions=[pred], references=[[pred]])

    # BLEU (default max_order=4) combines precisions with a *geometric mean*.
    # If ANY n-gram precision is 0, that mean is 0 → overall BLEU is 0 (even if 1–3 grams match).
    paraphrase_3gram = bleu.compute(
        predictions=[pred], references=[[ref_para]], max_order=3
    )
    paraphrase_smooth = bleu.compute(
        predictions=[pred], references=[[ref_para]], smooth=True
    )

    print("Paraphrase (max_order=4, default):", paraphrase)
    print("Paraphrase (max_order=3):", paraphrase_3gram)
    print("Paraphrase (smooth=True):", paraphrase_smooth)
    print("Identical:", identical)

Paraphrase (max_order=4, default): {'bleu': 0.0, 'precisions': [0.8333333333333334, 0.6, 0.25, 0.0], 'brevity_penalty': 0.846481724890614, 'length_ratio': 0.8571428571428571, 'translation_length': 6, 'reference_length': 7}
Paraphrase (max_order=3): {'bleu': 0.423240862445307, 'precisions': [0.8333333333333334, 0.6, 0.25], 'brevity_penalty': 0.846481724890614, 'length_ratio': 0.8571428571428571, 'translation_length': 6, 'reference_length': 7}
Paraphrase (smooth=True): {'bleu': 0.41386440336942737, 'precisions': [0.8571428571428571, 0.6666666666666666, 0.4, 0.25], 'brevity_penalty': 0.846481724890614, 'length_ratio': 0.8571428571428571, 'translation_length': 6, 'reference_length': 7}
Identical: {'bleu': 1.0, 'precisions': [1.0, 1.0, 1.0, 1.0], 'brevity_penalty': 1.0, 'length_ratio': 1.0, 'translation_length': 6, 'reference_length': 6}


In [20]:
try:
    import evaluate

    meteor = evaluate.load("meteor")
except Exception as e:
    meteor = None
    print("Skipping METEOR:", e)

if meteor is not None:
    try:
        import nltk

        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)
    except Exception as e:
        print("NLTK data:", e)

    # One reference per prediction: parallel lists of strings (see metric card for multi-ref list-of-lists).
    ex = EXAMPLES[1]
    identical_m = meteor.compute(predictions=[ex.reference], references=[ex.reference])
    qa_m = meteor.compute(predictions=[ex.prediction], references=[ex.reference])

    pred = "The cat sat on the mat"
    ref_para = "The cat is sitting on the mat"
    cat_m = meteor.compute(predictions=[pred], references=[ref_para])

    print("METEOR identical:", round(identical_m["meteor"], 4), identical_m)
    print("METEOR Tokyo QA (pred vs ref):", round(qa_m["meteor"], 4), qa_m)
    print("METEOR cat paraphrase:", round(cat_m["meteor"], 4), cat_m)

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/larryjin/nltk_data...
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/larryjin/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/larryjin/nltk_data...


METEOR identical: 0.5 {'meteor': np.float64(0.5)}
METEOR Tokyo QA (pred vs ref): 0.3571 {'meteor': np.float64(0.35714285714285715)}
METEOR cat paraphrase: 0.8535 {'meteor': np.float64(0.8534621578099838)}


## 4. Embedding metrics: SentenceTransformers cosine & BERTScore

**SentenceTransformers** maps each sentence to one dense vector. **Cosine similarity** between prediction and reference is a simple semantic similarity score—pick thresholds on a dev set; watch false positives on unrelated but similar-looking text.

**BERTScore** uses **token-level contextual embeddings** from a pretrained Transformer (BERT/RoBERTa/DeBERTa… via `transformers`), greedy matching, and optional IDF weights. It is **not** the same as comparing single sentence embeddings.

**Must it be “BERT”?** Only in spirit—the implementation lets you choose **`model_type`** (often RoBERTa or DeBERTa). It still needs **per-token hidden states** from a compatible checkpoint.

**Can BERTScore use SentenceTransformer embeddings?** **No** in the standard definition: ST produces **one vector per sentence**; BERTScore’s scoring needs **token vectors** and its matching steps. Using ST cosine similarity is a **different** (still useful) metric—call it sentence embedding similarity, not BERTScore.

Install: `sentence-transformers`, `bert-score`, and PyTorch. First run downloads model weights.

In [21]:
import numpy as np

preds = [ex.prediction for ex in EXAMPLES]
refs = [ex.reference for ex in EXAMPLES]


def cosine_sim_rows(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Row-wise cosine similarity for paired rows of a and b."""
    a_n = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-12)
    b_n = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
    return np.sum(a_n * b_n, axis=1)


# --- SentenceTransformers: one embedding per sentence, cosine pairs ---
try:
    from sentence_transformers import SentenceTransformer

    st_model = SentenceTransformer("all-MiniLM-L6-v2")
    pred_e = st_model.encode(preds, convert_to_numpy=True, show_progress_bar=False)
    ref_e = st_model.encode(refs, convert_to_numpy=True, show_progress_bar=False)
    st_sims = cosine_sim_rows(pred_e, ref_e)
    for ex, s in zip(EXAMPLES, st_sims, strict=True):
        print(ex.id, "ST cosine(pred, ref)", round(float(s), 4))
except ImportError as e:
    print("Skipping SentenceTransformers:", e)


# --- BERTScore: token-level embeddings + greedy match (Transformers, not ST) ---
try:
    import evaluate

    bertscore = evaluate.load("bertscore")
except Exception as e:
    bertscore = None
    print("Skipping BERTScore:", e)

if bertscore is not None:
    # Smaller checkpoint keeps downloads/runtime modest; swap for roberta-large etc. if needed.
    bs = bertscore.compute(
        predictions=preds,
        references=refs,
        lang="en",
        model_type="distilroberta-base",
    )
    for ex, p, r, f in zip(
        EXAMPLES, bs["precision"], bs["recall"], bs["f1"], strict=True
    ):
        print(
            ex.id,
            "BERTScore",
            "F1",
            round(float(f), 4),
            "P",
            round(float(p), 4),
            "R",
            round(float(r), 4),
        )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7640.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


capitals-1 ST cosine(pred, ref) 1.0
capitals-2 ST cosine(pred, ref) 0.7625
math-1 ST cosine(pred, ref) 0.8365


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19593.33it/s]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


capitals-1 BERTScore F1 1.0 P 1.0 R 1.0
capitals-2 BERTScore F1 0.8978 P 0.8884 R 0.9075
math-1 BERTScore F1 0.9288 P 0.9288 R 0.9288


## 5. LLM-as-judge — OpenAI + repo-root `.env`

Put **`OPENAI_API_KEY`** in **`.env` at the repository root** (same folder as `.gitignore`). The notebook loads it with **`python-dotenv`** by walking upward from the kernel’s current working directory until it finds `.env`—run Jupyter with cwd **`ml_evals/`** or the repo root so that file is discovered.

**The judge cell requires a valid key** (no offline mock). If `OPENAI_API_KEY` is missing after loading `.env`, the cell raises a clear error.

**Interview talking points:** rubric clarity, anchor examples, position bias (swap A/B), calibration, cost/latency, and validating judges against humans.

In [23]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv


def load_repo_dotenv() -> Path | None:
    """Load the first `.env` found walking up from cwd (repo root when running under `ml_evals/`)."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        candidate = directory / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return candidate
    return None


_env_path = load_repo_dotenv()
_api_key = (os.environ.get("OPENAI_API_KEY") or "").strip()
if not _api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is missing. Set it in the repo-root `.env` file or export it in your environment "
        "(see Section 5). "
        f"Dotenv file loaded: {_env_path!s}; cwd: {Path.cwd()!s}"
    )
if _env_path:
    print(f"Loaded env file: {_env_path}")

JUDGE_SYSTEM = """You grade short answers. Output ONLY valid JSON with keys:
correctness (0-1 float), helpfulness (0-1 float), reasoning (one sentence).
Use the reference answer as ground truth but allow paraphrases if factually equivalent."""


def build_judge_user_prompt(question: str, reference: str, prediction: str) -> str:
    return (
        f"Question: {question}\n"
        f"Reference answer: {reference}\n"
        f"Model answer: {prediction}\n"
        "Respond with JSON only."
    )


def parse_judge_json(raw: str) -> dict:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`").removeprefix("json").strip()
    return json.loads(raw)


def openai_judge_llm(user_prompt: str) -> str:
    from openai import OpenAI

    model = (os.environ.get("OPENAI_JUDGE_MODEL") or "gpt-4o-mini").strip()
    client = OpenAI(api_key=_api_key)
    resp = client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
    )
    return (resp.choices[0].message.content or "").strip()


def score_with_judge(ex: EvalExample) -> dict:
    user = build_judge_user_prompt(ex.question, ex.reference, ex.prediction)
    raw = openai_judge_llm(user)
    return parse_judge_json(raw)


[(ex.id, score_with_judge(ex)) for ex in EXAMPLES]

Loaded env file: /Users/larryjin/Documents/Programs/prototype/.env


[('capitals-1',
  {'correctness': 1,
   'helpfulness': 1,
   'reasoning': 'The model answer exactly matches the reference answer.'}),
 ('capitals-2',
  {'correctness': 1,
   'helpfulness': 1,
   'reasoning': 'The model answer is factually equivalent to the reference answer.'}),
 ('math-1',
  {'correctness': 0,
   'helpfulness': 0,
   'reasoning': 'The model answer is incorrect because 12 + 28 equals 40, not 41.'})]

## 6. Pairwise comparison sketch (A vs B, mitigate position bias)

Present **both** orders or randomize; aggregate wins / use Bradley–Terry or Elo for interview depth.

In [7]:
import random


def pairwise_prompt(question: str, ref: str, a: str, b: str, order: tuple[str, str]) -> str:
    first, second = order
    first_ans, second_ans = (a, b) if order == ("A", "B") else (b, a)
    return (
        f"Question: {question}\nReference (ground truth): {ref}\n"
        f"Answer {first}: {first_ans}\nAnswer {second}: {second_ans}\n"
        "Which answer is better overall (correctness + clarity)? Reply ONLY: A, B, or tie."
    )


def mock_pairwise_judge(prompt: str) -> str:
    """Demo judge: both answers mention Paris → tie (swap-safe). Replace with LLM API."""
    low = prompt.lower()
    if low.count("paris") >= 2:
        return "tie"
    return "A"


def evaluate_pairwise(question: str, ref: str, model_a: str, model_b: str, *, seed: int = 0) -> dict:
    rng = random.Random(seed)
    orders = [("A", "B"), ("B", "A")]
    rng.shuffle(orders)
    votes = []
    for order in orders:
        p = pairwise_prompt(question, ref, model_a, model_b, order)
        v = mock_pairwise_judge(p)
        votes.append((order, v))
    return {"votes": votes}


evaluate_pairwise(
    "Capital of France?",
    "Paris",
    model_a="Paris",
    model_b="paris.",
)

{'votes': [(('A', 'B'), 'tie'), (('B', 'A'), 'tie')]}

## 7. Aggregate metrics + bootstrap confidence interval

Shows you can discuss **uncertainty** on small eval sets — common senior-level follow-up.

In [25]:
def dataset_em_f1(examples: list[EvalExample]) -> dict[str, float]:
    ems = [exact_match(ex.prediction, ex.reference) for ex in examples]
    f1s = [token_f1(ex.prediction, ex.reference) for ex in examples]
    return {"exact_match": sum(ems) / len(ems), "token_f1_mean": sum(f1s) / len(f1s)}


def bootstrap_ci_mean(values: list[float], *, n_boot: int = 1000, seed: int = 0, alpha: float = 0.05):
    rng = np.random.default_rng(seed)
    arr = np.asarray(values, dtype=np.float64)
    n = len(arr)
    stats = []
    for _ in range(n_boot):
        sample = rng.choice(arr, size=n, replace=True)
        stats.append(float(sample.mean()))
    stats.sort()
    low = stats[int((alpha / 2) * n_boot)]
    high = stats[int((1 - alpha / 2) * n_boot) - 1]
    return {"mean": float(arr.mean()), "ci_low": low, "ci_high": high}


print("Point estimates:", dataset_em_f1(EXAMPLES))
f1_values = [token_f1(ex.prediction, ex.reference) for ex in EXAMPLES]
print("Bootstrap token-F1 mean CI:", bootstrap_ci_mean(f1_values))

Point estimates: {'exact_match': 0.3333333333333333, 'token_f1_mean': 0.4666666666666666}
Bootstrap token-F1 mean CI: {'mean': 0.4666666666666666, 'ci_low': 0.0, 'ci_high': 1.0}


## 8. Library cheat sheet — what to say in interviews

| Library | Typical use |
|---------|----------------|
| **`datasets`** | Load public benchmarks (HF); version splits. |
| **`evaluate`** | BLEU, ROUGE, METEOR, BERTScore, BLEURT wrappers; fast iteration. |
| **`ragas`** | Faithfulness, answer relevance, context precision/recall for RAG. |
| **`deepeval`** | Metric presets + LLM test cases in CI-ish workflows. |
| **LangSmith / Phoenix** | Trace runs, compare experiments, human annotation queues. |
| **OpenAI Evals** | YAML specs + grading patterns; good mental model for prod eval pipelines. |

**RAG mini-pattern:** retrieve → generate → score *faithfulness* (claim-level vs context) + *answer relevance* (vs question). Mention **chunking**, **top-k**, and **hallucination** checks.

---

### Next steps for you

1. Confirm **`OPENAI_API_KEY`** in repo-root **`.env`** for Section 5; optional **`OPENAI_JUDGE_MODEL`**.
2. Export `EXAMPLES` to JSONL and version in git; add adversarial cases (negation, dates, units).
3. Pick one public slice (e.g. subset of **TruthfulQA** or **GSM8K**) and run EM/F1 + one semantic metric.
4. Practice whiteboarding: draw data flow from **prompt → model → parser → metric → dashboard**.

In [ ]:
# Optional: Ragas needs API + retrieval contexts — https://docs.ragas.io
print("Notebook core sections complete.")